# 07｜手动替换图片数据集：从数字到服饰

前面六份 Notebook 使用 `load_digits` 的 8×8 灰度数字。本练习把图片来源换成文件夹中的 28×28 服饰照片，保留同一个 CNN 训练入口。你会亲手检查：**路径 → 类别名 → 标签编号 → 像素 → 四维张量 → 模型输出**。

先用项目自带的离线三类服饰包完整运行一次，再选两类重新运行，或接入你自己整理的图片。默认 CPU；不下载数据，也不需要 `torchvision`。建议用时 60–90 分钟。

In [ ]:
%matplotlib inline
from pathlib import Path
from zipfile import ZipFile
import sys
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch

LAB_DIR = Path.cwd()
if not (LAB_DIR / "cnn_lab").is_dir():
    LAB_DIR = LAB_DIR / "02_cnn_lab"
if not (LAB_DIR / "cnn_lab").is_dir():
    raise FileNotFoundError(f"找不到 cnn_lab；当前运行目录为 {Path.cwd()}。请在 VS Code 中打开 02_cnn_lab 或项目根目录，并从头运行。")
LAB_DIR = LAB_DIR.resolve()
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))
torch.set_num_threads(1)
from cnn_lab import ImageDatasetBundle, ExperimentConfig, SmallCNN, configure_chinese_font, train_experiment
print("运行目录：", Path.cwd())
print("实验目录：", LAB_DIR)
print("Python：", sys.executable)

## 第一步：找到并解压图片

课程包已经存放在 `data/fashion_small.zip`。首次运行会解压出 `data/fashion_small/`，其中 `train`、`val`、`test` 下分别是三个类别文件夹。你可以在 VS Code 文件浏览器中打开一张 PNG，确认目录名就是类别名。

以后使用自己的数据时，把 `DATA_DIR` 改为自己图片文件夹的路径。需要保持同样的 `train/类别名/图片文件`、`val/类别名/图片文件`、`test/类别名/图片文件` 结构。

In [ ]:
ARCHIVE = LAB_DIR / "data" / "fashion_small.zip"
DATA_DIR = LAB_DIR / "data" / "fashion_small"  # 唯一需要修改的数据路径
if not DATA_DIR.is_dir():
    if DATA_DIR != LAB_DIR / "data" / "fashion_small":
        raise FileNotFoundError(f"找不到自定义数据目录 {DATA_DIR}；请检查 DATA_DIR")
    if not ARCHIVE.is_file():
        raise FileNotFoundError(f"找不到数据目录 {DATA_DIR}，也找不到课程压缩包 {ARCHIVE}")
    with ZipFile(ARCHIVE) as archive:
        expected_prefix = "fashion_small/"
        if any(not name.startswith(expected_prefix) or ".." in Path(name).parts for name in archive.namelist()):
            raise ValueError("压缩包路径异常，请重新获取课程文件")
        archive.extractall(LAB_DIR / "data")
print("数据目录：", DATA_DIR)
for split in ("train", "val", "test"):
    folder = DATA_DIR / split
    if not folder.is_dir():
        raise FileNotFoundError(f"缺少 {folder}；请检查 DATA_DIR")
    print(split, sorted(p.name for p in folder.iterdir() if p.is_dir()))

## 第二步：建立类别与标签的对应关系

模型需要整数标签，文件夹使用文字类别名。下面的 `CLASS_NAMES` 决定编号：`bag → 0`、`sneaker → 1`、`trousers → 2`。三个划分必须使用**同一组顺序**。换成自己的数据时，在这里改类别名；不要分别为三个划分生成不同的编号。

**动手检查：**三类分别有多少张？如果一个文件夹缺失，应该先修正数据目录，不要继续训练。

In [ ]:
CLASS_NAMES = ("bag", "sneaker", "trousers")  # 接入自己数据时，在这里改类别名
IMAGE_SIZE = 28  # 当前 SmallCNN 处理正方形灰度图
class_to_label = {name: index for index, name in enumerate(CLASS_NAMES)}
print("类别 → 标签：", class_to_label)
for split in ("train", "val", "test"):
    folder = DATA_DIR / split
    found = {p.name for p in folder.iterdir() if p.is_dir()}
    missing = set(CLASS_NAMES) - found
    if missing:
        raise FileNotFoundError(f"{split} 缺少类别文件夹：{sorted(missing)}")
    counts = {name: sum(path.suffix.lower() in {".png", ".jpg", ".jpeg"} for path in (folder / name).iterdir() if path.is_file()) for name in CLASS_NAMES}
    if any(count == 0 for count in counts.values()):
        raise ValueError(f"{split} 中存在空类别：{counts}")
    print(split, counts)

## 第三步：把一张图片变成模型输入

图片文件里的像素是 0–255 的整数。模型输入需要 `float32`，范围 0–1，还要增加通道维度：`[H,W] → [1,H,W]`。原数字实验的像素范围是 0–16，那里使用“除以 16”；这里要**除以 255**。

**动手检查：**运行下方单元格，解释四维 `[B,C,H,W]` 中每个字母的含义。

In [ ]:
example_path = sorted(p for p in (DATA_DIR / "train" / CLASS_NAMES[0]).iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"})[0]
with Image.open(example_path) as image:
    gray = image.convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
    pixels = np.asarray(gray, dtype=np.float32)
normalized = pixels / 255.0
one_image = torch.from_numpy(normalized[None, :, :])
print("文件：", example_path.name)
print("原像素形状和范围：", pixels.shape, (pixels.min(), pixels.max()))
print("CNN 单张输入形状和范围：", tuple(one_image.shape), (float(one_image.min()), float(one_image.max())))
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(pixels, cmap="gray_r", vmin=0, vmax=255); axes[0].set_title("文件中的灰度图")
axes[1].imshow(one_image[0], cmap="gray_r", vmin=0, vmax=1); axes[1].set_title("归一化后的输入")
for axis in axes: axis.axis("off")
plt.show()

## 第四步：读取三个划分

下面的 `read_split()` 逐个读取 `split/类别名/图片`，把文件夹名转换为同一套整数标签。请先读懂循环，再运行。代码同时接受 PNG/JPG/JPEG；彩色文件会转成灰度。对自己拍摄的照片，缩放可能损失细节，要在报告中说明。

原官方训练集中的样本分给 `train` 和 `val`；`test` 来自官方独立测试集。不要把同一张图片复制到两个划分，更不要看测试结果后反复选方案。

In [ ]:
def read_split(split: str) -> tuple[torch.Tensor, torch.Tensor]:
    pictures, labels = [], []
    for class_name, label in class_to_label.items():
        folder = DATA_DIR / split / class_name
        files = sorted(p for p in folder.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"})
        if not files:
            raise FileNotFoundError(f"{folder} 中没有可用图片")
        for path in files:
            try:
                with Image.open(path) as image:
                    gray = image.convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
                    array = np.asarray(gray, dtype=np.float32) / 255.0
            except Exception as error:
                raise ValueError(f"无法读取图片 {path}: {error}") from error
            pictures.append(array[None, :, :])  # [H,W] → [C,H,W]，C=1
            labels.append(label)
    x = torch.from_numpy(np.stack(pictures))  # [N,1,H,W]
    y = torch.tensor(labels, dtype=torch.long)  # 交叉熵需要整数类别编号
    return x, y

x_train, y_train = read_split("train")
x_val, y_val = read_split("val")
x_test, y_test = read_split("test")
data = ImageDatasetBundle(
    x_train=x_train, y_train=y_train, x_val=x_val, y_val=y_val,
    x_test=x_test, y_test=y_test, class_names=CLASS_NAMES,
    dataset_name=DATA_DIR.name, pixel_range=(0.0, 1.0),
)
for name, x, y in (("train", x_train, y_train), ("val", x_val, y_val), ("test", x_test, y_test)):
    print(name, "图片", tuple(x.shape), "标签", tuple(y.shape), "范围", f"{x.min():.2f}–{x.max():.2f}")
    assert x.shape[1:] == (1, IMAGE_SIZE, IMAGE_SIZE)
    assert y.dtype == torch.long and set(y.tolist()) == set(range(len(CLASS_NAMES)))
assert len(set(CLASS_NAMES)) == len(CLASS_NAMES)
probe = SmallCNN(image_size=data.image_size, n_classes=data.n_classes, channels=(4, 8))
print("一个 batch 的模型输出：", tuple(probe(x_train[:4]).shape))
assert probe(x_train[:4]).shape == (4, len(CLASS_NAMES))

## 第五步：复用已有 CNN 训练

现在的数据对象与原数字实验使用同一个 `ImageDatasetBundle` 接口。训练时每轮观察 **验证准确率**；训练完成后才计算一次 **测试准确率**。先使用下面的 CPU 配置完整跑通，不以分数高低排名。

In [ ]:
config = ExperimentConfig(
    name="三类服饰文件夹实验", channels=(4, 8), kernel_size=3,
    pooling="max", learning_rate=0.001, batch_size=64,
    epochs=8, seed=42, device="cpu",
)
result = train_experiment(config, data=data)
assert result.val_accuracy is not None and len(result.val_accuracy) == config.epochs
assert len(result.test_accuracy) == 1
print(f"最后一轮验证准确率：{result.val_accuracy[-1]:.1%}")
print(f"一次测试准确率：{result.final_test_accuracy:.1%}")
print(f"可训练参数量：{result.parameter_count:,}")

## 第六步：用图核查结果

先看 Loss 和训练/验证准确率，再看测试集混淆矩阵。图中类别名称应与你设定的 `CLASS_NAMES` 一致。测试图片只用来描述最终错误；需要修改模型或参数时，应回到训练集与验证集重新设计实验。

In [ ]:
configure_chinese_font()
epochs = np.arange(1, config.epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(epochs, result.train_loss, marker="o"); axes[0].set(xlabel="Epoch", ylabel="Cross-Entropy Loss", title="训练 Loss")
axes[1].plot(epochs, result.train_accuracy, marker="o", label="训练")
axes[1].plot(epochs, result.val_accuracy, marker="o", label="验证")
axes[1].set(xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1.02), title="训练与验证准确率")
axes[1].legend(); plt.show()

result.model.eval()
with torch.no_grad():
    prediction = result.model(data.x_test).argmax(dim=1).cpu()
confusion = np.zeros((data.n_classes, data.n_classes), dtype=int)
for truth, guess in zip(data.y_test.tolist(), prediction.tolist()):
    confusion[truth, guess] += 1
fig, axis = plt.subplots(figsize=(5, 4))
axis.imshow(confusion, cmap="Blues")
axis.set_xticks(range(data.n_classes), data.class_names, rotation=30)
axis.set_yticks(range(data.n_classes), data.class_names)
axis.set(xlabel="预测类别", ylabel="真实类别", title="测试集混淆矩阵")
for row in range(data.n_classes):
    for column in range(data.n_classes):
        axis.text(column, row, str(confusion[row, column]), ha="center", va="center")
plt.show()
print("各类正确数：", {name: int(confusion[i, i]) for i, name in enumerate(CLASS_NAMES)})

## 第七步：自己替换一次

**必做迁移：**把 `CLASS_NAMES` 改为上面三类中的两类，从“第二步”重新运行到第六步。记录模型输出为什么从 `[B,3]` 变成 `[B,2]`，以及三个划分中每类图片数量。不要只修改最终输出层；要确认数据和标签也一起改变。

**进阶迁移：**自己整理一个小型灰度图分类集，放在 `data/我的数据/train|val|test/类别名/图片`。类别要互斥，拍摄同一实物的近似照片放在同一划分中。然后修改 `DATA_DIR` 与 `CLASS_NAMES`，从头运行。若图片不是正方形，当前代码会统一缩放为 `IMAGE_SIZE × IMAGE_SIZE`。

### 我的记录（请自行填写）

- 数据路径、类别映射：
- `train / val / test` 每类张数：
- 单张与 batch 的形状、像素范围：
- 哪些代码需要随数据更换？
- 验证与测试结果，以及混淆矩阵中一个具体错误：
- 本次结论的适用范围与下一步：

**自查：**路径真实存在；三个划分类别一致；标签连续且从 0 开始；输入 `[B,1,H,W]`；输出 `[B,K]`；测试集只用于最后检查。

## 完成后清理

下方单元格会关闭图片并结束当前内核；运行后 VS Code 显示“内核已停止”属于正常现象。需要重新实验时再选择课程内核。

In [ ]:
%reset -f
import gc
import matplotlib.pyplot as plt
import torch
from IPython import get_ipython
plt.close("all")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("清理完成，正在结束 Notebook 内核……")
get_ipython().kernel.do_shutdown(restart=False)